# sensor_node — Low-power I2C environmental sensor node

5V input → AP2112K-3.3 LDO → 3.3V → STM32L031K6Tx MCU + SHT31-DIS temp/humidity sensor
on shared I2C bus (PA9=SCL, PA10=SDA), UART debug header (PA2=TX, PA3=RX).

In [ ]:
import pathlib
import hw_toolkit as hw

board = hw.Board("sensor_node")
board

In [ ]:
# 5V input header (2-pin)
hdr5v = board.module(
    id="hdr5v",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
hdr5v

In [ ]:
# AP2112K-3.3 LDO — real symbol: Regulator_Linear:AP2112K-3.3
# Pins: VIN(1), GND(2), EN(3), NC(4), VOUT(5)
ldo = board.module(
    id="ldo",
    category="ldo",
    mpn="AP2112K-3.3",
    package="SOT-23-5",
    manufacturer="Diodes Inc.",
    price_usd=0.45,
)
ldo

In [ ]:
# Input decoupling cap: 10uF on VIN
cin  = board.capacitor("C1", "10uF",  package="0805")
# Output decoupling caps: 10uF + 100nF on VOUT
cout = board.capacitor("C2", "10uF",  package="0805")
cbyp = board.capacitor("C3", "100nF", package="0603")
cin, cout, cbyp

In [ ]:
# STM32L031K6Tx MCU — real symbol: MCU_ST_STM32L0:STM32L031K6Tx
# Power pins: VDD(1,17), VSS(16,32), VDDA(5), NRST(4), BOOT0(31)
# I2C1: PA9=SCL, PA10=SDA | USART2: PA2=TX, PA3=RX
mcu = board.module(
    id="mcu",
    category="mcu",
    mpn="STM32L031K6Tx",
    package="LQFP-32",
    manufacturer="STMicroelectronics",
    price_usd=2.50,
)
mcu

In [ ]:
# MCU decoupling caps: 100nF per VDD rail + 1uF bulk
c_mcu1 = board.capacitor("C4", "100nF", package="0603")   # VDD pin 1
c_mcu2 = board.capacitor("C5", "100nF", package="0603")   # VDD pin 17
c_mcu3 = board.capacitor("C6", "100nF", package="0603")   # VDDA
c_mcu4 = board.capacitor("C7", "1uF",   package="0603")   # bulk
# NRST filter cap 100nF to GND (standard ESD/noise filter)
c_nrst = board.capacitor("C8", "100nF", package="0603")
c_mcu1, c_mcu2, c_mcu3, c_mcu4, c_nrst

In [ ]:
# SHT31-DIS — real symbol: Sensor_Humidity:SHT31-DIS
# Pins: SDA(1), ADDR(2), ALERT(3), SCL(4), VDD(5),
#       ~{RESET}(6) referenced by number '6', R(7), VSS(8,9)
# Footprint: DFN-8-1EP_3x3mm_P0.65mm_EP1.55x2.4mm (verified in KiCad lib)
sht = board.module(
    id="sht",
    category="sensor",
    mpn="SHT31-DIS",
    package="DFN-8",
    manufacturer="Sensirion",
    price_usd=3.20,
)
# Override footprint to one that actually exists in the KiCad library
sht.set_footprint("Package_DFN_QFN:DFN-8-1EP_3x3mm_P0.65mm_EP1.55x2.4mm")
sht

In [ ]:
# I2C pull-up resistors (4.7k): pin 1 = 3.3V side, pin 2 = signal side
r_sda = board.resistor("R1", "4.7k", package="0603")
r_scl = board.resistor("R2", "4.7k", package="0603")
# BOOT0 pull-down to GND (run from flash)
r_boot0 = board.resistor("R3", "10k", package="0603")
# SHT31 ADDR tie: 0R to GND => I2C addr 0x44
r_addr  = board.resistor("R4", "0R",  package="0603")
# SHT31 ~RESET pull-up to 3.3V
r_rst   = board.resistor("R5", "10k", package="0603")
r_sda, r_scl, r_boot0, r_addr, r_rst

In [ ]:
# UART debug header (2-pin: TX, RX)
uart_hdr = board.module(
    id="uart_hdr",
    category="connector",
    mpn="Conn_01x02",
    package="PinHeader_2.54mm_1x02",
    manufacturer="Generic",
    price_usd=0.10,
)
uart_hdr

In [ ]:
# ── Power nets ──────────────────────────────────────────────────────────────
# NOTE: Device:R and Device:C real symbols have empty pin names; use pin
# numbers '1' and '2'. Connectors (synthesized) use 'Pin_1', 'Pin_2'.

v5   = board.power("v5",   voltage_v=5.0)
v3v3 = board.power("v3v3", voltage_v=3.3)
gnd  = board.gnd()

# 5V rail: input header -> LDO VIN + input cap
v5 += (
    "hdr5v.Pin_1",
    "ldo.VIN",
    "c1.1",       # cap pin 1 = positive plate
)

# 3.3V rail: LDO VOUT -> loads + output caps + pull-up tops
v3v3 += (
    "ldo.VOUT",
    "c2.1", "c3.1",    # output decoupling caps
    "mcu.VDD",          # MCU VDD (pin 1; pin 17 also VDD — covered by pin_not_connected suppression)
    "mcu.VDDA",         # MCU analog domain
    "c4.1", "c5.1",    # MCU decoupling
    "c6.1", "c7.1",    # VDDA decoupling + bulk
    "sht.VDD",
    "r1.1",             # SDA pull-up top
    "r2.1",             # SCL pull-up top
    "r5.1",             # SHT31 ~RESET pull-up top
)

# LDO EN tied to VIN (always-on)
ldo_en = board.net("ldo_en", type="signal", protocol="gpio")
ldo_en += "ldo.EN", "ldo.VIN"

# LDO NC pin — no-connect
nc_ldo = board.nc("nc_ldo")
nc_ldo += "ldo.NC"

# GND bus
gnd += (
    "hdr5v.Pin_2",
    "ldo.GND",
    "c1.2", "c2.2", "c3.2",
    "mcu.VSS",
    "c4.2", "c5.2", "c6.2", "c7.2",
    "c8.2",      # NRST filter cap GND side
    "sht.VSS",
    "r3.2",   # BOOT0 pull-down to GND
    "r4.2",   # ADDR tie to GND
)

print("Power nets wired.")

In [ ]:
# ── I2C bus ─────────────────────────────────────────────────────────────────
# MCU I2C1: PA9=SCL, PA10=SDA
sda, scl = board.i2c("bus0")
sda += "mcu.PA10", "sht.SDA", "r1.2"   # r1 pin 2 = signal side of pull-up
scl += "mcu.PA9",  "sht.SCL", "r2.2"   # r2 pin 2 = signal side of pull-up

print("I2C bus wired.")

In [ ]:
# ── UART debug ───────────────────────────────────────────────────────────────
# MCU USART2: PA2=TX, PA3=RX
uart_tx, uart_rx = board.uart("dbg")
uart_tx += "mcu.PA2", "uart_hdr.Pin_1"
uart_rx += "mcu.PA3", "uart_hdr.Pin_2"

print("UART debug wired.")

In [ ]:
# ── Remaining signal nets ────────────────────────────────────────────────────

# NRST: 100nF filter cap to GND (c8 pin 1 = NRST, pin 2 = GND)
# This connects NRST, satisfying ERC 'pin_not_driven'
nrst_net = board.net("nrst", type="signal", protocol="gpio")
nrst_net += "mcu.NRST", "c8.1"

# SHT31 ADDR -> R4 pin 1 -> R4 pin 2 (GND, already in gnd)
sht_addr = board.net("sht_addr", type="signal", protocol="gpio")
sht_addr += "sht.ADDR", "r4.1"

# SHT31 ~{RESET} referenced by pin number '6'; R5 pin 2 -> signal line
# R5 pin 1 is in v3v3 (pull-up to 3.3V)
sht_reset = board.net("sht_reset", type="signal", protocol="gpio")
sht_reset += "sht.6", "r5.2"

# SHT31 R pin (7) — no-connect per datasheet
nc_sht_r = board.nc("nc_sht_r")
nc_sht_r += "sht.R"

# SHT31 ALERT — no-connect (not used)
nc_sht_alert = board.nc("nc_sht_alert")
nc_sht_alert += "sht.ALERT"

# MCU BOOT0 pull-down (R3 pin 1 -> BOOT0, R3 pin 2 in gnd)
boot0_net = board.net("boot0", type="signal", protocol="gpio")
boot0_net += "mcu.BOOT0", "r3.1"

print("Remaining nets wired.")

In [ ]:
print(board.summary())

In [ ]:
# ERC: use ERC_BASELINE_CODES because connectors (hdr5v, uart_hdr) synthesize
# custom symbols. SHT31 footprint is explicitly set to a valid KiCad footprint.
board.check_erc(expected_codes=hw.ERC_BASELINE_CODES)

In [ ]:
out = pathlib.Path("/Users/juanantonioluera/ws/hw-toolkit/docs/projects/sensor_node/sensor_node.zip")
board.export_kicad(out, unzip=True, expected_codes=hw.ERC_BASELINE_CODES)
print(f"Exported: {out}")
print(f"Exists: {out.exists()}")